In [ ]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import json
import os
class Config:
    DATA_PATH = os.getenv('DATA_PATH')
    MODEL_PATH = os.getenv('MODEL_PATH')
    
    MODEL_NAME = 'bert-base-uncased'  
    MAX_LEN = 128
    BATCH_SIZE = 16  
    EPOCHS = 8
    LEARNING_RATE = 2e-5
    
    NUM_WORKERS = 0
    PIN_MEMORY = False
    USE_AMP = True
    FREEZE_BERT_LAYERS = 6  
    PATIENCE = 2
    
    os.makedirs(MODEL_PATH, exist_ok=True)

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

usecols = ['text', 'age_group']
train_df = pd.read_csv(f'{config.DATA_PATH}/author_train_clean.csv', usecols=usecols)
val_df = pd.read_csv(f'{config.DATA_PATH}/author_val_clean.csv', usecols=usecols)
test_df = pd.read_csv(f'{config.DATA_PATH}/author_test_clean.csv', usecols=usecols)

age_to_id = {'Adult': 0, 'Minor': 1}
train_df['label'] = train_df['age_group'].map(age_to_id)
val_df['label'] = val_df['age_group'].map(age_to_id)
test_df['label'] = test_df['age_group'].map(age_to_id)

train_df = train_df.dropna().reset_index(drop=True)
val_df = val_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

print("Tokenizing...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

def tokenize_data(df, name):
    texts = df['text'].tolist()
    all_input_ids = []
    all_attention_masks = []
    
    for i in tqdm(range(0, len(texts), 1000), desc=f"  {name}", leave=False):
        batch_texts = texts[i:i+1000]
        tokens = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=config.MAX_LEN,
            return_tensors='pt'
        )
        all_input_ids.append(tokens['input_ids'])
        all_attention_masks.append(tokens['attention_mask'])
    
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_mask = torch.cat(all_attention_masks, dim=0)
    
    return {'input_ids': input_ids, 'attention_mask': attention_mask}

train_tokens = tokenize_data(train_df, "Train")
val_tokens = tokenize_data(val_df, "Val")
test_tokens = tokenize_data(test_df, "Test")

train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)
val_labels = torch.tensor(val_df['label'].values, dtype=torch.long)
test_labels = torch.tensor(test_df['label'].values, dtype=torch.long)

class AuthorDataset(Dataset):
    def __init__(self, tokens, labels):
        self.input_ids = tokens['input_ids']
        self.attention_mask = tokens['attention_mask']
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'label': self.labels[idx]
        }

train_dataset = AuthorDataset(train_tokens, train_labels)
val_dataset = AuthorDataset(val_tokens, val_labels)
test_dataset = AuthorDataset(test_tokens, test_labels)

del train_df, val_df, test_df, train_tokens, val_tokens, test_tokens
gc.collect()

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, 
                          num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE * 2, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE * 2, shuffle=False,
                         num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)

model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)


if config.FREEZE_BERT_LAYERS > 0:
    for layer in model.bert.encoder.layer[:config.FREEZE_BERT_LAYERS]:
        for param in layer.parameters():
            param.requires_grad = False

model = model.to(device)

train_labels_np = train_dataset.labels.numpy()
unique, counts = np.unique(train_labels_np, return_counts=True)
class_weights = torch.tensor([1.0/c for c in counts], dtype=torch.float).to(device)
class_weights = class_weights / class_weights.sum() * len(unique)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * config.EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                             num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.cuda.amp.GradScaler() if config.USE_AMP and device.type == 'cuda' else None

def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="    Training", leave=False)
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        else:
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()
        
        pbar.set_postfix({'loss': f'{total_loss/len(loader):.3f}', 'acc': f'{correct/total:.3f}'})
    
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="    Evaluating", leave=False)
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'acc': f'{correct/total:.3f}'})
    
    return total_loss / len(loader), correct / total, all_preds, all_labels

print("Training...")
best_val_acc = 0
no_improve = 0
id_to_age = {v: k for k, v in age_to_id.items()}

for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'accuracy': val_acc,
            'age_mapping': age_to_id,
            'id_to_age': id_to_age
        }, f'{config.MODEL_PATH}/best_model.pt')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= config.PATIENCE:
            print(f"  Early stopping")
            break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

checkpoint = torch.load(f'{config.MODEL_PATH}/best_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_preds, test_labels_list = evaluate(model, test_loader, criterion)

accuracy = accuracy_score(test_labels_list, test_preds)
f1 = f1_score(test_labels_list, test_preds, average='weighted')

print(f"\nTest Results:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1 Score: {f1:.4f}")

results = {
    'test_accuracy': float(accuracy),
    'test_f1_weighted': float(f1),
    'best_val_accuracy': float(best_val_acc)
}

with open(f'{config.MODEL_PATH}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

Device: cuda
Train: 430828 | Val: 108013 | Test: 41733
Tokenizing...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training...

Epoch 1/8


  Train Loss: 0.5894 | Acc: 0.8329
  Val Loss: 0.5192 | Acc: 0.6748

Epoch 2/8


  Train Loss: 0.5396 | Acc: 0.8156
  Val Loss: 0.4917 | Acc: 0.9234

Epoch 3/8


  Train Loss: 0.4863 | Acc: 0.8047
  Val Loss: 0.4975 | Acc: 0.8028

Epoch 4/8


  Train Loss: 0.4203 | Acc: 0.8290
  Val Loss: 0.5574 | Acc: 0.8402
  Early stopping

Best validation accuracy: 0.9234



Test Results:
  Accuracy: 0.9179
  F1 Score: 0.9067
